In [1]:
# -----------------------------------------------------
# Imports
# -----------------------------------------------------
from dash import Dash, dcc, html, dash_table, jupyter_dash
from dash.dependencies import Input, Output
import requests
import pandas as pd

/Users/dashrath/Documents/GA4GH/Analytics/analytics-dashboard/pypi/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# -----------------------------------------------------
# Configurations
# -----------------------------------------------------
BASE_URL = "http://127.0.0.1:8080/pypi"

# Endpoints
TOTAL_PACKAGES_ENDPOINT = f"{BASE_URL}/total-packages"
PACKAGE_VERSIONS_ENDPOINT = f"{BASE_URL}/package-versions"
RELEASES_OVER_YEARS_ENDPOINT = f"{BASE_URL}/releases-over-years"
ALL_SOURCES_COVERAGE_ENDPOINT = f"{BASE_URL}/all_sources_coverage"
PYPI_DETAILS_ENDPOINT = f"{BASE_URL}/project_details"

# Function to get JSON from API
def get_json(endpoint):
    resp = requests.get(endpoint)
    resp.raise_for_status()
    return resp.json()

In [3]:
# -----------------------------------------------------
# 1. Total packages
# -----------------------------------------------------
total_packages_resp = get_json(TOTAL_PACKAGES_ENDPOINT)
total_packages = int(total_packages_resp.get("total_packages", 0))

# -----------------------------------------------------
# 2. Project details
# -----------------------------------------------------
details_data = get_json(PYPI_DETAILS_ENDPOINT)

# Build DataFrame
rows = []
for pkg in details_data:
    versions_list = pkg.get("versions", [])
    versions_count = versions_list[0].get("versions", 0) if versions_list else 0
    rows.append({
        "project_name": pkg.get("project_name"),
        "description": pkg.get("description"),
        "author_name": pkg.get("author_name"),
        "author_email": pkg.get("author_email"),
        "category": pkg.get("category"),
        "versions_count": versions_count,
    })

df = pd.DataFrame(rows)

In [4]:
# -----------------------------------------------------
# 3. Dynamic datatable for high level view
# -----------------------------------------------------
app = Dash(__name__)

app.layout = html.Div([
    html.H2(
        f"Total PyPI Packages: {total_packages}",
        style={'textAlign': 'center', 'margin-bottom': '20px', 'color': '#2C3E50'}
    ),

    # Main Search Box (keystroke search)
    dcc.Input(
        id='table-search',
        type='text',
        placeholder='Search projects...',
        debounce=False,  # keystroke-based search
        style={
            'margin-bottom': '15px',
            'width': '350px',
            'padding': '8px',
            'border-radius': '5px',
            'border': '1px solid #ccc'
        }
    ),

    # Interactive DataTable
    dash_table.DataTable(
        id='projects-table',
        columns=[{"name": i, "id": i} for i in df.reset_index().columns],
        data=df.reset_index().to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',  # native per-column filter
        style_table={'overflowX': 'auto', 'border': '1px solid #ccc', 'border-radius': '5px'},
        style_header={'backgroundColor': '#34495E', 'color': 'white', 'fontWeight': 'bold'},
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'minWidth': '100px', 'width': '150px', 'maxWidth': '250px',
            'whiteSpace': 'nowrap',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'border-bottom': '1px solid #ddd'
        },
        style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#f9f9f9'}]
    ),
])

# Main Search -> filter DataTable
@app.callback(
    Output('projects-table', 'data'),
    Input('table-search', 'value')
)
def update_table(search_value):
    if not search_value:
        return df.reset_index().to_dict('records')
    filtered = df.reset_index()[df.reset_index().apply(
        lambda row: row.astype(str).str.contains(search_value, case=False).any(), axis=1
    )]
    return filtered.to_dict('records')


In [5]:
# -----------------------------------------------------
# 4. Dynamic bar graph inheriting data from datatable
# -----------------------------------------------------

app.layout.children.append(
    dcc.Graph(id="datatable-bar")
)

@app.callback(
    Output("datatable-bar", "figure"),
    Input("projects-table", "derived_virtual_data"),
    Input("projects-table", "page_current"),
    Input("projects-table", "page_size")
)

def update_bar(rows, page_current, page_size):
    dff = pd.DataFrame(rows) if rows else df.copy()

    # If "project_name" missing, rename index
    if "project_name" not in dff.columns:
        if "index" in dff.columns:
            dff = dff.rename(columns={"index": "project_name"})

    # Provide safe defaults
    page_current = page_current or 0
    page_size = page_size or 10

    start = page_current * page_size
    end = start + page_size
    page_data = dff.iloc[start:end]

    # Create hover text: shorten long description
    hover_texts = [
        f"<b>{row['project_name']}</b><br>"
        f"Category: {row.get('category', '')}<br>"
        f"Versions: {str(row.get('versions_count', ''))}"
        for _, row in page_data.iterrows()
    ]

    fig = {
        "data": [
            {
                "x": page_data["project_name"],
                "y": page_data["versions_count"],
                "type": "bar",
                "marker": {"color": "#2E86C1"},
                "hovertext": hover_texts,
                "hoverinfo": "text"
            }
        ],
        "layout": {
            "title": {
                "text": "Package Versions Count (Current Page)",
                "x": 0.5,          # center title
                "xanchor": "center",
                "yanchor": "top",
                "font": {"size": 20, "color": "#2C3E50"}
            },
            "xaxis": {"title": "project_name", "tickangle": -45},
            "yaxis": {"title": "versions_count"},
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff",
            "margin": {"b": 120}
        }
    }
    return fig

In [6]:
# -----------------------------------------------------
# 5. Category distribution
# -----------------------------------------------------
app.layout.children.append(
    dcc.Graph(id="category-distribution")
)

@app.callback(
    Output("category-distribution", "figure"),
    Input("projects-table", "derived_virtual_data")
)
def update_category_distribution(rows):
    dff = pd.DataFrame(rows) if rows is not None else df.reset_index()

    if "category" not in dff.columns or dff.empty:
        return {"data": [], "layout": {"title": "No category data available"}}

    # Count categories
    cat_counts = dff["category"].value_counts().reset_index()
    cat_counts.columns = ["category", "count"]

    return {
        "data": [{
            "labels": cat_counts["category"],
            "values": cat_counts["count"],
            "type": "pie",
            "hole": 0.4,
            "textinfo": "label+percent",
            "hoverinfo": "label+value+percent"
        }],
        "layout": {
            "title": {
                "text": "Category Distribution",
                "x": 0.5,          # center title
                "xanchor": "center",
                "yanchor": "top",
                "font": {"size": 20, "color": "#2C3E50"}
            },
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff"
        }
    }
    return fig

In [7]:
app.run()

In [10]:
# -----------------------------------------------------
# 6. Releases over years
# -----------------------------------------------------
app.layout.children.append(
    dcc.Graph(id="releases-over-years")
)

@app.callback(
    Output("releases-over-years", "figure"),
)
def releases_over_years(rows):
    releases = get_json(RELEASES_OVER_YEARS_ENDPOINT)
    dff = pd.DataFrame(releases)
    print(dff)
    return {
        "data": [
            {
                "x": dff["year"],
                "y": dff["releases"],
                "type": "bar",
                "marker": {"color": "#27AE60"},
                "hovertemplate": "%{x}: %{y} releases<extra></extra>"
            }
        ],
        "layout": {
            "title": {
                "text": "PyPI Releases Per Year",
                "x": 0.5,
                "xanchor": "center",
                "font": {"size": 18, "color": "#2C3E50"}
            },
            "xaxis": {"title": "Year"},
            "yaxis": {"title": "Number of Releases"},
            "plot_bgcolor": "#f9f9f9",
            "paper_bgcolor": "#ffffff"
        }
    }
    return fig

